# Train and Register the Model

In this notebook, you will build a lightweight baseline model for taxi trip duration, inspect what the engineered features look like, and register the finished pipeline in MLflow.

Before you run the notebook, complete [01-setup-local-stack.md](01-setup-local-stack.md) and keep the local MLflow, Prefect, and Postgres services running.

The goal is not to build the most accurate model possible, but to create a clear, repeatable baseline that can be reused by the batch and online prediction lessons.


## Workflow

```mermaid
flowchart LR
    A["Load taxi<br>Parquet data"] --> B["Prepare route and<br>distance features"]
    B --> C["Train regression<br>pipeline"]
    C --> D["Compare against a<br>simple baseline"]
    D --> E["Log run and model<br>to MLflow"]
```


In [ ]:
import mlflow
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

from src.common.config import get_settings
from src.common.features import (
    TARGET_COLUMN,
    load_dataframe,
    prepare_dataframe,
    to_model_records,
)
from src.common.model_registry import configure_tracking, ensure_experiment

settings = get_settings()
settings

## Load and Inspect the Training Data

We start with one Parquet file from the public NYC Green Taxi dataset. The helper function below keeps the notebook aligned with the active project code.

In [ ]:
df_raw = load_dataframe(settings.train_data_uri)
df_raw.head()

In [ ]:
prepared = prepare_dataframe(df_raw, include_target=True)
prepared[["trip_route", "trip_distance", TARGET_COLUMN]].head()

The model sees only two inputs:

- `trip_route`, a combined categorical feature built from pickup and dropoff IDs;
- `trip_distance`, a numeric distance feature.

That makes the baseline easy to understand and easy to reuse later.

In [ ]:
records = to_model_records(prepared)
target = prepared[TARGET_COLUMN]

X_train, X_valid, y_train, y_valid = train_test_split(
    records,
    target,
    test_size=0.2,
    random_state=42,
)

len(X_train), len(X_valid)

## Train the Baseline Pipeline

The pipeline combines `DictVectorizer` with `LinearRegression`. This is a good baseline because the preprocessing and model are visible in one object.

In [ ]:
pipeline = make_pipeline(DictVectorizer(), LinearRegression())
pipeline.fit(X_train, y_train)

predictions = pipeline.predict(X_valid)
rmse = root_mean_squared_error(y_valid, predictions)
print(f"RMSE: {rmse:.2f}")

## Add One Useful Comparison

A printed RMSE is more informative when we compare it against a very simple reference. Here we compare the trained model to a naive predictor that always uses the training set mean duration.

In [ ]:
mean_baseline = pd.Series([y_train.mean()] * len(y_valid), index=y_valid.index)
comparison = pd.DataFrame(
    {
        "model": ["mean baseline", "linear regression pipeline"],
        "rmse": [
            root_mean_squared_error(y_valid, mean_baseline),
            rmse,
        ],
    }
)
comparison["improvement_vs_baseline"] = comparison["rmse"].iloc[0] - comparison["rmse"]
comparison

## Visual Check: Residual Pattern

A residual plot helps confirm that the trained pipeline is doing more than just memorizing the average duration. If the points are centered around zero without an obvious trend, the baseline is behaving reasonably for this example.

In [ ]:
import matplotlib.pyplot as plt

residual_frame = pd.DataFrame(
    {
        "actual_duration": y_valid,
        "predicted_duration": predictions,
    }
)
residual_frame["residual"] = (
    residual_frame["actual_duration"] - residual_frame["predicted_duration"]
)

ax = residual_frame.plot.scatter(
    x="predicted_duration",
    y="residual",
    alpha=0.2,
    figsize=(7, 4),
    title="Residuals vs. Predicted duration",
)
ax.axhline(0, color="black", linestyle="--", linewidth=1)
plt.show()

A lower RMSE means the predictions are closer to the observed trip durations. If the linear model beats the mean baseline by a visible margin, it is strong enough for the rest of the repo.

## Register the Model in MLflow

The next cell logs the validation metric and registers the fitted pipeline so the batch flow and API can reuse it without retraining.

In [ ]:
configure_tracking(settings.mlflow_tracking_uri)
experiment_id = ensure_experiment(settings.experiment_name)
mlflow.set_experiment(settings.experiment_name)

with mlflow.start_run(experiment_id=experiment_id, run_name="notebook-baseline") as run:
    mlflow.log_param("feature_columns", "trip_route,trip_distance")
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("rmse", float(rmse))
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        name=settings.model_artifact_path,
        registered_model_name=settings.registered_model_name,
        serialization_format="skops",
    )
    run_summary = {
        "run_id": run.info.run_id,
        "experiment_name": settings.experiment_name,
        "registered_model_name": settings.registered_model_name,
        "rmse": float(rmse),
    }

run_summary

## Wrap Up

You have now registered a model in MLflow. If you ran the sanity check in [01-setup-local-stack.md](01-setup-local-stack.md), `green-taxi-duration` now has two versions: **version 1** from that check and **version 2** from this notebook. The batch flow and the API both resolve the latest version, so they will use version 2.